In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
#1) Orders schema
ordersSchema=StructType([
    StructField("order_id",IntegerType()),
    StructField("user_id", IntegerType()),
    StructField("order_time",StringType())
])
#Payments Schema
PaymentsSchema=StructType([
    StructField('payment_id',IntegerType()),
    StructField('order_id',IntegerType()),
    StructField('payment_time',StringType())
])

In [0]:
from pyspark.sql.functions import to_timestamp, col, expr

In [0]:
ordersRaw=(
    spark.readStream.format("csv")
    .schema(ordersSchema)
    .option("header",True)
    .option("maxFilesPerTrigger",1)
    .load("/mnt/input-data/streams/orders/"))

paymentsRaw=(
    spark.readStream.format("csv")
    .schema(PaymentsSchema)
    .option("header",True)
    .option("maxFilesPerTrigger",1)
    .load("/mnt/input-data/streams/payments/"))

In [0]:
ordersRaw.display()

In [0]:
paymentsRaw.display()

In [0]:
orders=ordersRaw.withColumn("order_time",to_timestamp(col("order_time"),"yyyy-MM-dd'T'HH:mm:ssX"))
payments=paymentsRaw.withColumn("payment_time",to_timestamp(col("payment_time"),"yyyy-MM-dd'T'HH:mm:ssX"))

In [0]:
#applying water mark and alias each df
ordersWM=orders.withWatermark("order_time","30 Minutes").alias("orders")
paymentsWM=payments.withWatermark("payment_time","30 Minutes").alias("payments")

In [0]:
#join using those aliases
joined=ordersWM.join(
    paymentsWM,
    expr("""
         orders.order_id=payments.order_id AND
         payments.payment_time >= orders.order_time AND
         payments.payment_time <= (orders.order_time + INTERVAL 30 MINUTES)
         """),
    how="inner"
)

In [0]:
display(joined)

In [0]:
joined.writeStream \
    .format("csv") \
    .outputMode("append") \
    .option("path", "/mnt/input-data/streaming/orders-payments") \
    .trigger(processingTime="10 seconds") \
    .option('checkpointLocation', '/mnt/input-data/streaming/checkpoints/orders-payments').start()

In [0]:
# display(
#   joined.writeStream
#         .format("memory")              # in-memory sink
#         .queryName("joined_events")    # name your temp table
#         .outputMode("append")
#         .start()
# )



#query.awaitTermination()